In [ ]:
import pandas as pd
from silver_staging_utils import connect_to_postgres, read_table
import numpy as np

In [ ]:
conn = connect_to_postgres()
if conn:
    df = read_table("SELECT * FROM silver.categories LIMIT 100", conn)
    conn.close()
    

In [ ]:
df.head()

### Products

In [ ]:
query = '''
SELECT
    *
FROM silver.products
WHERE
    snapshot_date IN (
        SELECT MAX(snapshot_date) FROM silver.products  
    )
'''
print(query)

In [ ]:
conn = connect_to_postgres()
if conn:
    df = read_table(query, conn)
    conn.close()

In [ ]:
df.info()

In [ ]:
df['snapshot_date'].unique()

In [ ]:
df['supermarket'].unique()

#### Final Price creation

Price problem with Casa Rica

In [ ]:
df["price"] = (
    df["price"]
    .astype(str)
    .str.replace(r"[^\d,\.]", "", regex=True)
    .str.replace(".", "", regex=False) 
    .str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

Coalesce to build final price

In [ ]:
df.loc[df['promotion_price'] == '0', 'promotion_price'] = None

In [ ]:
df["final_price"] = (
    df["promotion_price"]
        .combine_first(df['price'])
)

Validations

In [ ]:
df.loc[df['supermarket'] == 'casarica', 'price'].head()

In [ ]:
df['promotion_price'].unique()

In [ ]:
# Casa Rica get numeric
df.loc[df['supermarket'] == 'casarica', ['price', 'promotion_price', 'final_price']]

In [ ]:
# Coalesce
df.loc[(df['supermarket'] == 'biggie') & (df['promotion_price'].notna()), ['price', 'promotion_price', 'final_price']]

In [ ]:
df.loc[df['supermarket'] == 'real', ['price', 'promotion_price', 'final_price']]